In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [5]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [6]:
shap_values = explainer(prompts)

In [7]:
print(shap_values.values.squeeze())

[-3.50681455e-06  2.76400235e+00 -8.72354661e-01  2.00342429e+00
  4.72185472e+00  1.18334559e+00  1.11659276e-01  2.30677287e+00
  9.67523081e-01  5.83311160e+00  2.01544604e+00]


In [8]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze())
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [9]:
# For now just do it on the web UI since we don't have a dataset yet

## ASG Pipeline

### Prune

In [21]:

json_path = "temp_graph_files/austin.json"
source_set = 'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0, 0, 0, 0, 1/3, 0, 0, 1/3, 0, 1/3, 0]
kept_ids, pruned_adj, node_inf, node_rel, attr, metadata = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.6,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.8,
    edge_relevance_threshold=0.9,
    keep_all_tokens_and_logits=False,
)

In [22]:
print(len(kept_ids))

15


In [23]:
for i, node_id in enumerate(kept_ids):
    print(node_id, attr[node_id].get("clerp", ""), node_inf[i].item(), node_rel[i].item())

0_1861_9 Texas 0.6002651453018188 0.7890620231628418
1_72774_10 Code/licensing snippets 0.5339171290397644 0.7596157789230347
16_89970_9 Texas 0.5723963379859924 0.5983902812004089
20_44686_9 Texas locations/legal contexts 0.5682745575904846 0.4728766977787018
20_44686_10 Texas locations/legal contexts 0.5273332595825195 0.5067331790924072
22_11998_10 Texas and Dallas 0.5599021315574646 0.47572407126426697
E_18143_1 Emb: " Fact" 0.4144315719604492 0.41368937492370605
E_714_3 Emb: "  The" 0.4942685067653656 0.5043306350708008
E_6037_4 Emb: "  capital" 0.4552099406719208 0.150369331240654
E_576_5 Emb: "  of" 0.5032409429550171 0.6407560706138611
E_2329_7 Emb: "  state" 0.5198944807052612 0.4575140178203583
E_10751_8 Emb: "  containing" 0.48224300146102905 0.6810780763626099
E_26865_9 Emb: "  Dallas" 0.38820645213127136 0.11313237994909286
E_603_10 Emb: "  is" 0.4362860918045044 0.49940696358680725
27_22605_10 Output " Austin" (p=0.439) 0.30819204449653625 0.2774704098701477


### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [24]:
feature_types = classify_features(kept_ids, attr, metadata)
print(feature_types)

{'0_1861_9': 'input', '1_72774_10': 'trash', '16_89970_9': 'output', '20_44686_9': 'output', '20_44686_10': 'output', '22_11998_10': 'output', 'E_18143_1': 'embedding', 'E_714_3': 'embedding', 'E_6037_4': 'embedding', 'E_576_5': 'embedding', 'E_2329_7': 'embedding', 'E_10751_8': 'embedding', 'E_26865_9': 'embedding', 'E_603_10': 'embedding', '27_22605_10': 'logit'}


In [14]:
attr

{'0_1861_9': {'node_id': '0_1861_9',
  'feature': 1734452,
  'layer': '0',
  'ctx_idx': 9,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '0_1861-0',
  'clerp': 'Texas',
  'influence': 0.44640499353408813,
  'activation': 1.875},
 '1_72774_10': {'node_id': '1_72774_10',
  'feature': 2648209474,
  'layer': '1',
  'ctx_idx': 10,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '1_72774-0',
  'clerp': 'Code/licensing snippets',
  'influence': 0.32853609323501587,
  'activation': 11.6875},
 '16_89970_9': {'node_id': '16_89970_9',
  'feature': 4048875061,
  'layer': '16',
  'ctx_idx': 9,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '16_89970-0',
  'clerp': 'Texas',
  'influence': 0.40106865763664246,
 